In [1]:
from xgboost import XGBClassifier

print("XGBoost works")

XGBoost works


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Required libraries loaded")

Required libraries loaded


In [3]:
def evaluate_model(model_name, y_test, y_pred, y_prob):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

print("Evaluation function ready")

Evaluation function ready


In [21]:
import os
import pandas as pd

base_path = "../datasets"

projects = ["ant", "camel", "jedit", "poi", "xalan"]

all_dataframes = []

for project in projects:

    folder = os.path.join(base_path, project)

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".csv")
    ]

    for file in files:

        file_path = os.path.join(folder, file)

        df_temp = pd.read_csv(file_path)

        df_temp["project"] = project
        df_temp["version"] = file.replace(".csv", "")

        all_dataframes.append(df_temp)

combined_df = pd.concat(
    all_dataframes,
    ignore_index=True
)

combined_df["bug_binary"] = (
    combined_df["bug"] > 0
).astype(int)

print("Combined dataset shape:", combined_df.shape)
print(combined_df["bug_binary"].value_counts())

C:\Users\nazli\software_defect_prediction


FileNotFoundError: [WinError 3] The system cannot find the path specified: '../datasets\\ant'

In [5]:
feature_columns = [
    'wmc', 'dit', 'noc', 'cbo', 'rfc',
    'lcom', 'ca', 'ce', 'npm', 'lcom3',
    'loc', 'dam', 'moa', 'mfa', 'cam',
    'ic', 'cbm', 'amc', 'max_cc', 'avg_cc'
]

train_projects = ["camel", "jedit", "poi", "xalan"]
test_project = "ant"

train_df = combined_df[
    combined_df["project"].isin(train_projects)
]

test_df = combined_df[
    combined_df["project"] == test_project
]

X_train = train_df[feature_columns]
y_train = train_df["bug_binary"]

X_test = test_df[feature_columns]
y_test = test_df["bug_binary"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

Train shape: (9231, 20)
Test shape: (1692, 20)

Train class distribution:
bug_binary
0    5853
1    3378
Name: count, dtype: int64

Test class distribution:
bug_binary
0    1342
1     350
Name: count, dtype: int64


In [6]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred,
    rf_prob
)

rf_results

{'Model': 'Random Forest',
 'Accuracy': 0.7511820330969267,
 'Precision': 0.40273972602739727,
 'Recall': 0.42,
 'F1-score': 0.4111888111888112,
 'ROC-AUC': 0.7355886736214605}

In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(y_train.value_counts())
print()
print(y_train_smote.value_counts())

bug_binary
0    5853
1    3378
Name: count, dtype: int64

bug_binary
0    5853
1    5853
Name: count, dtype: int64


In [8]:
rf_smote_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_smote_model.fit(X_train_smote, y_train_smote)

rf_smote_pred = rf_smote_model.predict(X_test)
rf_smote_prob = rf_smote_model.predict_proba(X_test)[:, 1]

rf_smote_results = evaluate_model(
    "Random Forest + SMOTE",
    y_test,
    rf_smote_pred,
    rf_smote_prob
)

rf_smote_results

{'Model': 'Random Forest + SMOTE',
 'Accuracy': 0.7080378250591016,
 'Precision': 0.3771331058020478,
 'Recall': 0.6314285714285715,
 'F1-score': 0.4722222222222222,
 'ROC-AUC': 0.7276708537364275}

In [9]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train_smote, y_train_smote)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

xgb_results = evaluate_model(
    "XGBoost + SMOTE",
    y_test,
    xgb_pred,
    xgb_prob
)

xgb_results

{'Model': 'XGBoost + SMOTE',
 'Accuracy': 0.7021276595744681,
 'Precision': 0.3694915254237288,
 'Recall': 0.6228571428571429,
 'F1-score': 0.46382978723404256,
 'ROC-AUC': 0.7163764104747712}

In [10]:
rf_prob[:10]

array([0.455     , 0.4719881 , 0.16661111, 0.51283333, 0.307     ,
       0.25021429, 0.4110119 , 0.40591667, 0.37880556, 0.47808333])

In [11]:
xgb_prob[:10]

array([0.53787136, 0.59439266, 0.03888799, 0.60816187, 0.40666068,
       0.39833656, 0.24949177, 0.41216436, 0.25529635, 0.3936112 ],
      dtype=float32)

In [12]:
ensemble_prob = (rf_prob + xgb_prob) / 2

ensemble_pred = (ensemble_prob >= 0.5).astype(int)

ensemble_results = evaluate_model(
    "RF + XGB Ensemble",
    y_test,
    ensemble_pred,
    ensemble_prob
)

ensemble_results

{'Model': 'RF + XGB Ensemble',
 'Accuracy': 0.7411347517730497,
 'Precision': 0.412,
 'Recall': 0.5885714285714285,
 'F1-score': 0.48470588235294115,
 'ROC-AUC': 0.7331498829039812}

In [13]:
weighted_prob = (
    0.7 * rf_prob +
    0.3 * xgb_prob
)

weighted_pred = (weighted_prob >= 0.5).astype(int)

weighted_results = evaluate_model(
    "Weighted Ensemble",
    y_test,
    weighted_pred,
    weighted_prob
)

weighted_results

{'Model': 'Weighted Ensemble',
 'Accuracy': 0.7440898345153665,
 'Precision': 0.41113490364025695,
 'Recall': 0.5485714285714286,
 'F1-score': 0.4700122399020808,
 'ROC-AUC': 0.7367990206514797}

In [14]:
results_df = pd.DataFrame([
    rf_results,
    rf_smote_results,
    xgb_results,
    ensemble_results,
    weighted_results
])

results_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Random Forest,0.751182,0.402740,0.420000,0.411189,0.735589
1,Random Forest + SMOTE,0.708038,0.377133,0.631429,0.472222,0.727671
2,XGBoost + SMOTE,0.702128,0.369492,0.622857,0.463830,0.716376
3,RF + XGB Ensemble,0.741135,0.412000,0.588571,0.484706,0.733150
4,Weighted Ensemble,0.744090,0.411135,0.548571,0.470012,0.736799


In [15]:
results_df.to_csv("results_summary.csv", index=False)

print("Results saved")

Results saved


In [16]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(10)

,Feature,Importance
10,loc,0.124577
17,amc,0.096027
3,cbo,0.069987
4,rfc,0.065335
14,cam,0.059358
6,ca,0.058629
9,lcom3,0.058629
13,mfa,0.052578
7,ce,0.051991
19,avg_cc,0.051296


In [18]:
from sklearn.model_selection import ParameterGrid

rf_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5],
    "class_weight": [None, "balanced"]
}

rf_tuning_results = []

for params in ParameterGrid(rf_param_grid):
    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train_smote, y_train_smote)
    
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    
    result = evaluate_model(
        "Tuned Random Forest",
        y_test,
        pred,
        prob
    )
    
    result.update(params)
    rf_tuning_results.append(result)

rf_tuning_df = pd.DataFrame(rf_tuning_results)
rf_tuning_df.sort_values(by="F1-score", ascending=False).head(10)

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,class_weight,max_depth,min_samples_leaf,n_estimators
49,Tuned Random Forest,0.736407,0.415789,0.677143,0.515217,0.756036,balanced,10.0,2,300
13,Tuned Random Forest,0.736407,0.415789,0.677143,0.515217,0.756036,NaN,10.0,2,300
12,Tuned Random Forest,0.731087,0.409949,0.682857,0.512326,0.753175,NaN,10.0,2,200
48,Tuned Random Forest,0.731087,0.409949,0.682857,0.512326,0.753175,balanced,10.0,2,200
53,Tuned Random Forest,0.734634,0.413005,0.671429,0.511425,0.757514,balanced,10.0,5,500
17,Tuned Random Forest,0.734634,0.413005,0.671429,0.511425,0.757514,NaN,10.0,5,500
14,Tuned Random Forest,0.733452,0.411867,0.674286,0.511376,0.756689,NaN,10.0,2,500
50,Tuned Random Forest,0.733452,0.411867,0.674286,0.511376,0.756689,balanced,10.0,2,500
11,Tuned Random Forest,0.731087,0.408056,0.665714,0.505972,0.756771,NaN,10.0,1,500
47,Tuned Random Forest,0.731087,0.408056,0.665714,0.505972,0.756771,balanced,10.0,1,500


In [19]:
best_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_smote, y_train_smote)

best_prob = best_rf.predict_proba(X_test)[:, 1]

In [20]:
from sklearn.metrics import f1_score

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    
    pred = (best_prob >= threshold).astype(int)

    print(
        threshold,
        round(f1_score(y_test, pred), 4)
    )

0.3 0.3828
0.35 0.4191
0.4 0.4584
0.45 0.4892
0.5 0.5152
0.55 0.5146
0.6 0.4559
